# Revision Questions: Data Understanding & Data Privacy

The data we will work on contains the same kinds of "messiness" that exist in real-world data:
* missing engagement metrics (for "protected" accounts)
* a date column stored as text
* a users table with duplicate rows
* a sentiment file with a messy/mixed dtype
* raw category labels that need to be simplified
* usernames/mentions that should be handled carefully from a privacy perspective

**Files (in the `da3_data` folder):**
* `synthetic_tweets.csv` — one row per tweet
* `synthetic_users_raw.csv` — one row per user *appearance* (authors + mentioned users, so it has duplicates)
* `synthetic_sentiment.csv` — sentiment scores (`positive`, `negative`, `neutral`, SentiStrength-style trinary) for a subset of tweets

## Setup

In [1]:
import pandas as pd
import re

## Question 1 — Load and inspect

Load `synthetic_tweets.csv` into a dataframe called `tweets`. Check its shape, column names, and dtypes.

**Note:** Before working with any data you need to know what you're actually working with. Checking shape/columns/dtypes catches loading mistakes early (wrong file, wrong separator, wrong encoding), problems that are much harder to debug once they've fed into later steps.

**Hint:**
```python
tweets = pd.read_csv('da3_data/synthetic_tweets.csv')
tweets.shape
tweets.dtypes
```

In [5]:
# Accessing function documentation without leaving JupyterLab

# 1. help() - the full docstring, printed below this cell. Jupyter.
# help(pd.read_csv)

# # 2. The '?' shortcut - same info as help(), but opens in its own pane (doesn't clutter your notebook)
# pd.read_csv?

# # 3. Shift+Tab - put your cursor inside the parentheses below and press Shift+Tab
# pd.read_csv()

# # this is how you look up documentation for attributes
help(pd.DataFrame.shape)      

Help on property:

    Return a tuple representing the dimensionality of the DataFrame.

    Unlike the `len()` method, which only returns the number of rows, `shape`
    provides both row and column counts, making it a more informative method for
    understanding dataset size.

    See Also
    --------
    numpy.ndarray.shape : Tuple of array dimensions.

    Examples
    --------
    >>> df = pd.DataFrame({"col1": [1, 2], "col2": [3, 4]})
    >>> df.shape
    (2, 2)

    >>> df = pd.DataFrame({"col1": [1, 2], "col2": [3, 4], "col3": [5, 6]})
    >>> df.shape
    (2, 3)



In [7]:
!ls

NOTEBOOK1_merging_guest_lists_STUDENT.ipynb
NOTEBOOK2_merge_issues_SOLUTIONS.ipynb
NOTEBOOK2_merge_issues_STUDENT.ipynb
da3_data
da3_data_json
guestlist_ally.csv
guestlist_bert.pkl
wtt_w3_inclass_SOLUTION.ipynb
wtt_w3_json_and_concepts.ipynb


In [9]:
# ANSWER ()
# tweets = pd.read_csv('inclass/da3_data/synthetic_tweets.csv')
tweets = pd.read_csv('da3_data/synthetic_tweets.csv')
print(tweets.shape)
print(tweets.columns.tolist())
print(tweets.dtypes)
tweets.head()

(450, 11)
['tweet_id', 'author_id', 'created_at', 'text', 'lang', 'source', 'topic_raw', 'retweet_count', 'reply_count', 'like_count', 'quote_count']
tweet_id           int64
author_id          int64
created_at           str
text                 str
lang                 str
source               str
topic_raw            str
retweet_count    float64
reply_count      float64
like_count       float64
quote_count      float64
dtype: object


,tweet_id,author_id,created_at,text,lang,source,topic_raw,retweet_count,reply_count,like_count,quote_count
0,1700000000000000000,900000000000000000,2024-08-13 00:39:00,Local government just announced new funding fo...,en,Twitter Web App,Carbon Emissions,3.0,4.0,15.0,1.0
1,1700000000000000091,900000000000001096,2024-09-06 21:58:00,RT @solarhub3140_19: @sunnytalks3920_48 My res...,de,Twitter for iPhone,Extreme Weather,4.0,4.0,8.0,2.0
2,1700000000000000182,900000000000004384,2024-08-03 11:34:00,Skeptical of the latest claims about flood def...,en,Twitter for iPhone,Climate Denial Debate,4.0,3.0,11.0,3.0
3,1700000000000000273,900000000000002329,2024-09-11 08:39:00,Op-ed: what we're getting wrong about wildlife...,fr,TweetDeck,Corporate Sustainability,NaN,NaN,NaN,NaN
4,1700000000000000364,900000000000002329,2024-08-16 14:36:00,@oceanaction9933_38 Honestly tired of hearing ...,es,Twitter Web App,Wildlife & Conservation,NaN,NaN,NaN,NaN


## Question 2 — Check for missing values

Which columns have missing values, and how many? Also check `.describe()` on the numeric columns.

**Note:** Missing values can silently break or bias calculations (e.g. an average computed while quietly ignoring 30% of rows). Finding them early means *you* decide what to do about them, instead of being surprised by a wrong result later.

**Hint:**
```python
tweets.isna().sum()
tweets.describe()
```

In [28]:
# ANSWER ()
tweets.isna().sum()

tweet_id           0
author_id          0
created_at         0
text               0
lang               0
source             0
topic_raw          0
retweet_count    125
reply_count      125
like_count       125
quote_count      125
dtype: int64

In [10]:
tweets.describe()

,tweet_id,author_id,retweet_count,reply_count,like_count,quote_count
count,4.500000e+02,4.500000e+02,325.000000,325.000000,325.000000,325.000000
mean,1.700000e+18,9.000000e+17,14.956923,5.720000,49.584615,1.987692
std,1.183451e+04,1.978279e+03,74.819730,14.083034,180.034675,3.087185
min,1.700000e+18,9.000000e+17,3.000000,2.000000,8.000000,1.000000
25%,1.700000e+18,9.000000e+17,3.000000,2.000000,10.000000,1.000000
50%,1.700000e+18,9.000000e+17,4.000000,3.000000,14.000000,1.000000
75%,1.700000e+18,9.000000e+17,8.000000,5.000000,25.000000,2.000000
max,1.700000e+18,9.000000e+17,1221.000000,201.000000,2502.000000,36.000000


## Question 3 — Investigate *why* values are missing

`retweet_count`, `reply_count`, `like_count`, `quote_count` are missing for the same rows. Check whether the missing values are concentrated in specific `author_id`s, or spread randomly.

**Note:** Not all missing data is the same. Randomly missing data behaves very differently from data missing for a systematic reason (e.g. certain accounts hide their stats).

**Hint:**
```python
missing = tweets[tweets['like_count'].isna()]
missing['author_id'].value_counts()
```

In [40]:
# ANSWER ()
missing = tweets[tweets['like_count'].isna()]
missing_authors = missing['author_id'].value_counts()
print(missing_authors)

# NOt logical
# # do these authors ever have non-missing rows too?
# ok_authors = set(tweets[tweets['like_count'].notna()]['author_id'].unique())
# overlap = set(missing_authors.index) & ok_authors
# print('authors with both missing and non-missing rows:', len(overlap))

author_id
900000000000002329    26
900000000000002192    10
900000000000006302     7
900000000000003699     5
900000000000004521     4
900000000000002466     4
900000000000000548     3
900000000000000685     2
900000000000004247     2
900000000000000822     1
Name: count, dtype: int64


In [42]:
missing.dtypes

tweet_id                  int64
author_id                 int64
created_at       datetime64[us]
text                        str
lang                        str
source                      str
topic_raw                   str
retweet_count           float64
reply_count             float64
like_count              float64
quote_count             float64
topic                       str
dtype: object

In [ ]:
df10 = df10[df10['author_id'] != '900000000000002329']       # remove rows for one specific author

## Question 4 — Handle the missing values

Decide whether to `fillna()` or `dropna()` the rows with missing engagement counts, and apply it.

**Note:** `dropna()` and `fillna()` produce different datasets and can lead to different conclusions. Picking one without a reason is a common way analyses might lead to errors.

**Hint:**
```python
tweets = tweets.dropna(subset=['retweet_count', 'reply_count', 'like_count', 'quote_count'])
# OR: tweets[['retweet_count', ...]] = tweets[['retweet_count', ...]].fillna(0)
```

In [33]:
# ANSWER ()
# Missing metrics belong entirely to a fixed set of authors (protected accounts) -
# the counts are unavailable, not genuinely zero, so we drop these rows rather than fillna(0).
tweets = tweets.dropna(subset=['retweet_count', 'reply_count', 'like_count', 'quote_count'])
# After droping you should see 0 for missing values
tweets.isna().sum()

tweet_id         0
author_id        0
created_at       0
text             0
lang             0
source           0
topic_raw        0
retweet_count    0
reply_count      0
like_count       0
quote_count      0
dtype: int64

## Question 5 — Fix the date column

`created_at` is stored as text. Convert it to datetime, then keep only tweets posted from 1 September 2024 onwards.

**Note:** A text column that *looks* like a date still behaves like text: you can't reliably filter, sort chronologically, or compute time differences on it. This is one of the most common real-world data analysis issue — always check dtype before assuming a date column.

**Hint:**
```python
tweets['created_at'] = pd.to_datetime(tweets['created_at'])
tweets = tweets[tweets['created_at'] > '2024-09-01']
```

In [41]:
tweets.columns

Index(['tweet_id', 'author_id', 'created_at', 'text', 'lang', 'source',
       'topic_raw', 'retweet_count', 'reply_count', 'like_count',
       'quote_count', 'topic'],
      dtype='str')

In [14]:
# Datatypes of the columns 
tweets.dtypes

tweet_id           int64
author_id          int64
created_at           str
text                 str
lang                 str
source               str
topic_raw            str
retweet_count    float64
reply_count      float64
like_count       float64
quote_count      float64
dtype: object

In [15]:
# ANSWER ()
tweets['created_at'] = pd.to_datetime(tweets['created_at'])
print(tweets.dtypes['created_at'])
tweets = tweets[tweets['created_at'] > '2024-09-01']
len(tweets)

datetime64[us]


208

In [16]:
tweets.dtypes

tweet_id                  int64
author_id                 int64
created_at       datetime64[us]
text                        str
lang                        str
source                      str
topic_raw                   str
retweet_count           float64
reply_count             float64
like_count              float64
quote_count             float64
dtype: object

## Question 6 — Recategorize a messy column

`tweets['topic_raw']` has many specific categories. Write a function that groups them into 4 broader buckets and apply it to create a new `topic` column.

**Note:** Real category columns are often too granular to analyze usefully (dozens of labels, some with only a handful of rows). Grouping them into a few meaningful buckets is what makes group comparisons (like Question 9) actually readable and meaningful.

**Hint:** (you can customize this function)
```python
def recategorize(topic):
    if topic in ['Climate Policy', 'Climate Denial Debate']:
        return 'Policy & Society'
    elif topic in ['Extreme Weather', 'Wildlife & Conservation']:
        return 'Environment & Nature'
    elif topic in ['Renewable Energy', 'Green Technology', 'Corporate Sustainability']:
        return 'Technology & Business'
    else:
        return 'Other'

tweets['topic'] = tweets['topic_raw'].apply(recategorize)
```

In [17]:
# ANSWER ()
def recategorize(topic):
    if topic in ['Climate Policy', 'Climate Denial Debate']:
        return 'Policy & Society'
    elif topic in ['Extreme Weather', 'Wildlife & Conservation']:
        return 'Environment & Nature'
    elif topic in ['Renewable Energy', 'Green Technology', 'Corporate Sustainability']:
        return 'Technology & Business'
    else:
        return 'Other'

In [19]:
tweets['topic'] = tweets['topic_raw'].apply(recategorize)
tweets['topic'].value_counts()

topic
Technology & Business    61
Other                    54
Environment & Nature     47
Policy & Society         46
Name: count, dtype: int64

## Question 7 — Load the sentiment file and fix its dtype

Load `synthetic_sentiment.csv` into `sentiment`. Check the dtypes of `positive`, `negative`, `neutral` — they should be numeric but aren't. Investigate why, then clean them.

**Note:** A column that 'looks numeric' can still load as text if even one value doesn't parse (a typo, a placeholder like 'unk'). If you don't catch this, `.mean()` and similar functions either error out or — worse — silently give you a meaningless answer.

**Hint:**
```python
sentiment['positive'].unique()  # look for non-numeric values
sentiment['positive'] = pd.to_numeric(sentiment['positive'], errors='coerce')
```

In [22]:
# ANSWER ()
sentiment = pd.read_csv('da3_data/synthetic_sentiment.csv')
print(sentiment.dtypes)
print(sentiment['positive'].unique()[:10])  # notice non-numeric entries

tweet_id    int64
positive      str
negative      str
neutral       str
dtype: object
<StringArray>
['4', '3', '1', '2', 'unk', '5']
Length: 6, dtype: str


In [24]:
sentiment['positive'] = pd.to_numeric(sentiment['positive'], errors='coerce')
sentiment['negative'] = pd.to_numeric(sentiment['negative'], errors='coerce')
sentiment['neutral'] = pd.to_numeric(sentiment['neutral'], errors='coerce')
sentiment.dtypes

tweet_id      int64
positive    float64
negative    float64
neutral     float64
dtype: object

In [25]:
help(pd.to_numeric)

Help on function to_numeric in module pandas:

to_numeric(arg, errors: 'DateTimeErrorChoices' = 'raise', downcast: "Literal['integer', 'signed', 'unsigned', 'float'] | None" = None, dtype_backend: 'DtypeBackend | lib.NoDefault' = <no_default>)
    Convert argument to a numeric type.

    If the input is already of a numeric dtype, the dtype will be preserved.
    For non-numeric inputs, the default return dtype is `float64` or `int64`
    depending on the data supplied. Use the `downcast` parameter
    to obtain other dtypes.

    Please note that precision loss may occur if really large numbers
    are passed in. Due to the internal limitations of `ndarray`, if
    numbers smaller than `-9223372036854775808` (np.iinfo(np.int64).min)
    or larger than `18446744073709551615` (np.iinfo(np.uint64).max) are
    passed in, it is very likely they will be converted to float so that
    they can be stored in an `ndarray`. These warnings apply similarly to
    `Series` since it internally leve

## Question 8 — Merge tweets and sentiment

Check `tweet_id` is unique in both dataframes, then merge `tweets` and `sentiment` on `tweet_id` using `how='left'`, `'right'`, `'inner'`, `'outer'` and compare the resulting lengths.

**Note:** If a merge key isn't unique, a merge can duplicate rows and inflate your dataset without throwing an error. Always check uniqueness first. And picking the wrong `how=` is one of the most common real-world data bugs: it either drops rows you needed or inflates count. Comparing lengths across all four is a fast sanity check — e.g. if `inner` is much smaller than `left`, a lot of your tweets simply don't have a sentiment score.

**Hint:**
```python
tweets['tweet_id'].is_unique, sentiment['tweet_id'].is_unique
for how in ['left', 'right', 'inner', 'outer']:
    print(how, len(tweets.merge(sentiment, on='tweet_id', how=how)))
```

In [26]:
# ANSWER ()
print(tweets['tweet_id'].is_unique, sentiment['tweet_id'].is_unique)

True True


In [27]:
for how in ['left', 'right', 'inner', 'outer']:
    print(how, len(tweets.merge(sentiment, on='tweet_id', how=how)))

merged = tweets.merge(sentiment, on='tweet_id', how='left')
len(merged)

left 208
right 390
inner 174
outer 424


208

## Question 9 — Group and summarize

Group `merged` by `topic` and compute mean and standard deviation of the engagement columns. Use `.transpose()` for readability.

**Note:** Grouping is how you analyze large data report your finding, like 'topic A gets more engagement than topic B'. 

**Hint:**
```python
merged.groupby('topic')[['like_count', 'retweet_count']].mean().transpose()
```

In [39]:
# ANSWER ()
engagement_cols = ['like_count', 'retweet_count', 'reply_count', 'quote_count']
print(merged.groupby('topic')[engagement_cols].mean().transpose())
print(merged.groupby('topic')[engagement_cols].std().transpose())

topic          Environment & Nature      Other  Policy & Society  \
like_count                69.161290  43.184211         41.085714   
retweet_count              5.645161  41.552632         10.485714   
reply_count                4.903226   4.421053          4.771429   
quote_count                2.483871   1.973684          2.171429   

topic          Technology & Business  
like_count                    22.650  
retweet_count                  6.475  
reply_count                    9.800  
quote_count                    2.425  
topic          Environment & Nature       Other  Policy & Society  \
like_count               173.002716  131.054941        128.669291   
retweet_count              3.028183  197.214069         25.228802   
reply_count                5.850099    5.611916          5.916506   
quote_count                6.276257    1.938052          2.216061   

topic          Technology & Business  
like_count                 38.620806  
retweet_count               5.134037  
r

## Question 10 — Aggregate sentiment score & sample check

Create one column `sentiment_compound` that combines `positive` and `negative` into a single score, describe it (mean/SD), then take a random sample of 15 rows and look at `text` next to the score.

**Note:** Combining several columns into one score is common (composite indices, summary scores) but it hides assumptions about how the pieces should combine. Manually checking a sample against the raw text is how you sanity-check that your formula actually behaves the way you think it does, before trusting it at scale.

**Hint:**
```python
merged['sentiment_compound'] = merged['positive'] + merged['negative']
merged.sample(15, random_state=1)[['text', 'sentiment_compound']]
```

In [28]:
# ANSWER ()
merged['sentiment_compound'] = merged['positive'] + merged['negative']
print(merged['sentiment_compound'].describe())

sample15 = merged.sample(15, random_state=1)[['text', 'sentiment_compound']]
sample15

count    167.000000
mean       0.017964
std        1.883579
min       -4.000000
25%       -1.000000
50%        0.000000
75%        1.000000
max        4.000000
Name: sentiment_compound, dtype: float64


,text,sentiment_compound
186,RT @stormreport7669_9: This graph on the youth...,1.0
155,My research this year has focused on carbon em...,3.0
165,Small businesses are adapting fast when it com...,-1.0
200,Kids today deserve better answers on flood def...,-1.0
58,Can we talk about how extreme weather affects ...,1.0
34,Honestly tired of hearing about flood defenses...,0.0
151,RT @ecovoice6940_30: Local government just ann...,0.0
18,@sunnypost10_12 Local government just announce...,NaN
202,@leaftalks7887_21 Proud to see my city investi...,-1.0
62,@solardesk4890_11 Just read a great thread on ...,NaN


## Question 11 — Understand the users table

Load `synthetic_users_raw.csv` into `users_raw`. Compare `len(users_raw)` to `users_raw['id'].nunique()` and check `.value_counts()` on `id` to see why they differ.

**Note:** Data pulled from APIs often comes with exactly this kind of structural duplication built in. If you don't recognize it, you'll miscount things (e.g. think you have more distinct users than you do) and later merges will behave unexpectedly.

**Hint:**
```python
users_raw = pd.read_csv('da3_data/synthetic_users_raw.csv')
len(users_raw), users_raw['id'].nunique()
users_raw['id'].value_counts()
```

In [31]:
# ANSWER ()
users_raw = pd.read_csv('da3_data/synthetic_users_raw.csv')
print(len(users_raw), users_raw['id'].nunique())
users_raw['id'].value_counts()

732 50


id
900000000000002329    51
900000000000000959    29
900000000000001370    25
900000000000006439    25
900000000000002740    20
900000000000002192    20
900000000000001096    19
900000000000004384    19
900000000000005069    19
900000000000001233    19
900000000000002603    19
900000000000002055    18
900000000000006028    18
900000000000001918    17
900000000000003151    16
900000000000001507    16
900000000000003836    15
900000000000006302    15
900000000000000411    15
900000000000003425    15
900000000000005754    14
900000000000000274    14
900000000000006165    14
900000000000004658    14
900000000000000000    13
900000000000005206    13
900000000000001644    13
900000000000005343    13
900000000000005480    13
900000000000002466    13
900000000000004932    12
900000000000005617    12
900000000000004521    12
900000000000002877    11
900000000000006713    11
900000000000004110    11
900000000000003699    11
900000000000000137    10
900000000000004795    10
900000000000006576    

## Question 12 — Create a unique users table

Create `users_unique` with exactly one row per user `id`.

**Note:** This is the fix for Question 11 — but `drop_duplicates()` can silently throw away rows you actually wanted if you don't specify the right `subset`. Always be clear on *what makes a row a duplicate* before you call it, not just the syntax.

**Hint:**
```python
users_unique = users_raw.drop_duplicates(subset=['id'])
```

In [32]:
# ANSWER ()
users_unique = users_raw.drop_duplicates(subset=['id'])
len(users_unique) == users_raw['id'].nunique()

True

## Question 13 — Data minimization

Merge `merged` with `users_unique` (left merge, tweets on the left) to add author info. Then create `df_min` keeping only columns relevant to: *does sentiment predict engagement, controlling for follower count?*

**Note:** Keeping every column makes a dataset harder to work with, and with personal data it's a privacy risk. Align your column selection to your actual research question is good practice generally.

**Hint:**
```python
df = merged.merge(users_unique, how='left', left_on='author_id', right_on='id', suffixes=('_tweet', '_user'))
df_min = df[['tweet_id', 'username', 'text', 'sentiment_compound', 'like_count', 'retweet_count', 'followers_count', 'verified']]
```

In [39]:
help(df.merge)

Help on method merge in module pandas.core.frame:

merge(right: 'DataFrame | Series', how: 'MergeHow' = 'inner', on: 'IndexLabel | AnyArrayLike | None' = None, left_on: 'IndexLabel | AnyArrayLike | None' = None, right_on: 'IndexLabel | AnyArrayLike | None' = None, left_index: 'bool' = False, right_index: 'bool' = False, sort: 'bool' = False, suffixes: 'Suffixes' = ('_x', '_y'), copy: 'bool | lib.NoDefault' = <no_default>, indicator: 'str | bool' = False, validate: 'MergeValidate | None' = None) -> 'DataFrame' method of pandas.DataFrame instance
    Merge DataFrame or named Series objects with a database-style join.

    A named Series object is treated as a DataFrame with a single named column.

    The join is done on columns or indexes. If joining columns on
    columns, the DataFrame indexes *will be ignored*. Otherwise if joining indexes
    on indexes or indexes on a column or columns, the index will be passed on.
    When performing a cross merge, no column specifications to merg

In [34]:
# ANSWER ()
df = merged.merge(users_unique, how='left', left_on='author_id', right_on='id', suffixes=('_tweet', '_user'))
df.columns

Index(['tweet_id', 'author_id', 'created_at', 'text', 'lang', 'source',
       'topic_raw', 'retweet_count', 'reply_count', 'like_count',
       'quote_count', 'topic', 'positive', 'negative', 'neutral',
       'sentiment_compound', 'id', 'username', 'description', 'verified',
       'protected', 'followers_count', 'following_count', 'tweet_count',
       'listed_count', 'location', 'account_created_at'],
      dtype='str')

In [35]:
df_min = df[['tweet_id', 'username', 'text', 'sentiment_compound',
             'like_count', 'retweet_count', 'followers_count', 'verified']]
df_min.head()

,tweet_id,username,text,sentiment_compound,like_count,retweet_count,followers_count,verified
0,1700000000000000091,urbandaily8086_8,RT @solarhub3140_19: @sunnytalks3920_48 My res...,1.0,8.0,4.0,156,False
1,1700000000000000273,bluedaily2168_17,Op-ed: what we're getting wrong about wildlife...,-2.0,NaN,NaN,356,False
2,1700000000000000819,sunnyaction8977_47,RT @carbondaily5068_40: Small businesses are a...,NaN,561.0,5.0,125,False
3,1700000000000000910,windylab453_37,Op-ed: what we're getting wrong about wildlife...,3.0,18.0,3.0,65,False
4,1700000000000001092,polarnow4340_7,"Just read a great thread on green technology, ...",1.0,21.0,3.0,76,False


## Question 14 — Pseudonymize the authors

`df_min` still has `username`. Build a lookup of unique usernames with a sequential `pseudoID`, merge it in, then delete `username`.

**Note:** Pseudonymization keeps analysis possible (you can still tell if the same person posted twice) while removing the direct identifier.

**Hint:**
```python
lookup = df_min[['username']].drop_duplicates().reset_index(drop=True).reset_index().rename(columns={'index': 'pseudoID'})
df_min = df_min.merge(lookup, how='left', on='username')
del df_min['username']
```

In [37]:
# ANSWER ()
lookup = df_min[['username']].drop_duplicates()
lookup['pseudoID'] = range(len(lookup))

In [ ]:
df_min = df_min.merge(lookup, how='left', on='username')
del df_min['username']
df_min.head()

## Question 15 — Anonymize mentions in the tweet text

Replace every `@username` mention inside `df_min['text']` with the placeholder `@mention`.

**Note:** Even after removing usernames from the users table, free text can still contain identifying information — here, other people's handles. This is a reminder that privacy risks hide in unstructured fields too, not just in obviously-named columns like `username`.

**Hint:**
```python
df_min['text'] = df_min['text'].replace(to_replace=r'@\S+', value='@mention', regex=True)
```

In [54]:
# ANSWER ()
df_min['text'] = df_min['text'].replace(to_replace=r'@\S+', value='@mention', regex=True)
df_min['text'].head()

0    RT @mention @mention My research this year has...
1    RT @mention Small businesses are adapting fast...
2    Op-ed: what we're getting wrong about wildlife...
3    Just read a great thread on green technology, ...
4    Can we talk about how wildlife conservation af...
Name: text, dtype: str

# Bonus Questions

These go beyond the core revision. They combine skills from multiple questions above and introduce a few new pandas tricks (`transform`, `pivot_table`, `merge(validate=...)`). They use `tweets`, `merged`, `users_raw`, `users_unique`, and `df` as defined in your answers above — run the questions above first.


## Bonus 1 — Detect retweets and compare engagement

Tweets that start with `RT @username:` are retweets, not original content. Create a boolean column `is_retweet` on `tweets`, then compare mean engagement (`like_count`, `retweet_count`) between retweets and originals.

**Note:** Retweets and original posts are fundamentally different content (an endorsement vs. an authored opinion). Lumping them together can quietly distort an analysis — here, retweets and originals actually have very different average engagement.

**Hint:**
```python
tweets['is_retweet'] = tweets['text'].str.match(r'^RT @\w+:')
tweets.groupby('is_retweet')[['like_count', 'retweet_count']].mean()
```

In [56]:
# ANSWER ()
tweets['is_retweet'] = tweets['text'].str.match(r'^RT @\w+:')
tweets['is_retweet'].value_counts()
tweets.groupby('is_retweet')[['like_count', 'retweet_count']].mean()

,like_count,retweet_count
is_retweet,,
False,31.401786,18.18750
True,81.625000,10.71875


## Bonus 2 — Extract and rank hashtags

Write a function that extracts all hashtags from a tweet's text as a list, apply it to create a `hashtags` column, then use `.explode()` to find the 10 most common hashtags overall.

**Note:** Regular expressions are the standard tool for pulling structured pieces (hashtags, mentions, emails, URLs) out of free text — a skill that transfers directly to almost any text dataset you'll work with, not just Twitter data.

**Hint:**
```python
def extract_hashtags(text):
    return re.findall(r'#\w+', text)

tweets['hashtags'] = tweets['text'].apply(extract_hashtags)
tweets.explode('hashtags')['hashtags'].value_counts().head(10)
```

In [57]:
# ANSWER ()
def extract_hashtags(text):
    return re.findall(r'#\w+', text)

tweets['hashtags'] = tweets['text'].apply(extract_hashtags)
top_hashtags = tweets.explode('hashtags')['hashtags'].value_counts().head(10)
top_hashtags

hashtags
#COP                19
#ClimateAction      18
#Environment        16
#ClimateCrisis      15
#ClimateJustice     11
#RenewableEnergy    11
#Sustainability     10
#GreenTech          10
#ClimateChange       8
#NetZero             6
Name: count, dtype: int64

## Bonus 3 — Follower-normalized engagement rate

Using `df` (tweets merged with user info), compute `engagement_rate = like_count / followers_count` for each tweet. Handle the case where `followers_count` is 0 (avoid dividing by zero). Then find the 5 authors with the highest *average* engagement rate.

**Note:** Raw counts favor big accounts; normalizing by followers lets you fairly compare a small account to a large one. Handling the divide-by-zero case is also a realistic reminder that real data has edge cases that will crash naive code if you don't plan for them.

**Hint:**
```python
df['engagement_rate'] = df['like_count'] / df['followers_count'].replace(0, pd.NA)
df.groupby('username')['engagement_rate'].mean().sort_values(ascending=False).head(5)
```

In [ ]:
# ANSWER ()
df['engagement_rate'] = df['like_count'] / df['followers_count'].replace(0, pd.NA)

top5 = df.groupby('username')['engagement_rate'].mean().sort_values(ascending=False).head(5)
top5